# Emotion Recognition (FER) Fine-Tuning (Colab)

Trains a **lightweight facial-emotion** classifier to replace DeepFace. DeepFace's
stock FER model is slow and weak on real frames; a fine-tuned **EfficientNet-B0**
runs comfortably on a 4 GB GPU and is far more accurate on the 7 emotions.

7 classes (FER-2013):  angry, disgust, fear, happy, neutral, sad, surprise

**Stress** is *derived* from the emotion probabilities (no separate model needed):
a weighted sum of the negative-affect emotions. The mapping is saved in the
checkpoint so inference stays consistent with training.

**Tuned for free Colab T4 + 4 GB local inference** (compact backbone, 160px input).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =====================================================================
#  CONFIG  -  after you upload the whole Fine_tuning folder to
#  MyDrive/VisionAI/, these paths already point at this model folder.
#  (dataset lives INSIDE it - see DATASETS.txt.)
# =====================================================================
BASE = "/content/drive/MyDrive/VisionAI/Fine_tuning/04_emotion_fer"

# FER-2013 layout:  <FER_ROOT>/train/<emotion>/*.jpg , <FER_ROOT>/test/<emotion>/*.jpg
FER_ROOT = BASE + "/fer2013"

WORK_DIR = BASE + "/_work"

# =====================================================================
#  HYPERPARAMETERS
# =====================================================================
MODEL_NAME = "efficientnet_b0"    # +acc vs mobilenet; still 4GB-OK at inference
IMG_SIZE   = 160
BATCH_SIZE = 256          # eff_b0@160: fits T4; lower to 128 if OOM
EPOCHS     = 60           # max per fold; early stopping ends sooner
LR         = 5e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.1
SEED       = 42

import os
os.makedirs(WORK_DIR, exist_ok=True)
print("Work dir:", WORK_DIR)

In [ ]:
!pip -q install timm
import torch, subprocess
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print(subprocess.getoutput("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader"))

## Data loaders
FER images are 48x48 grayscale; we upscale to `IMG_SIZE`, replicate to 3 channels
(so the ImageNet-pretrained backbone applies), and ImageNet-normalize. Training
augmentation: horizontal flip + small rotation + light affine.

In [ ]:
import os, numpy as np, torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler, Dataset
from sklearn.model_selection import train_test_split
from PIL import Image

torch.manual_seed(SEED); np.random.seed(SEED)
MEAN = [0.485, 0.456, 0.406]; STD = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(12),
    transforms.RandomAffine(0, translate=(0.06, 0.06), scale=(0.92, 1.08)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)])

# --- Stage FER-2013 to local SSD (Drive is too slow for 36k tiny files) ---
import shutil
LOCAL_ROOT = "/content/fer2013"
if not os.path.isdir(LOCAL_ROOT):
    print("Staging FER-2013 to local disk (one-time copy, a few minutes)...")
    shutil.copytree(FER_ROOT, LOCAL_ROOT)
    print("Staged ->", LOCAL_ROOT)
FER_ROOT = LOCAL_ROOT

train_dir = os.path.join(FER_ROOT, "train")
test_dir  = os.path.join(FER_ROOT, "test")
assert os.path.isdir(train_dir) and os.path.isdir(test_dir), "Check FER_ROOT (need train/ and test/)."

# Pool BOTH official splits (same class order), then a stratified 75/25 split:
# 75% train+val pool (K-fold CV runs here) and a 25% held-out TEST set.
CLASS_NAMES = datasets.ImageFolder(train_dir).classes
NUM_CLASSES = len(CLASS_NAMES)
samples = datasets.ImageFolder(train_dir).samples + datasets.ImageFolder(test_dir).samples
all_paths  = [p for p, _ in samples]
all_labels = np.array([l for _, l in samples])
tr_paths, te_paths, tr_lab, te_lab = train_test_split(
    all_paths, all_labels, test_size=0.25, random_state=SEED, stratify=all_labels)
tr_paths = np.array(tr_paths); te_paths = np.array(te_paths)
print("Classes:", CLASS_NAMES)
print("Pool:", len(all_paths), "| train+val:", len(tr_paths), " held-out test:", len(te_paths))

class FERDataset(Dataset):
    def __init__(self, paths, labels, tf):
        self.paths = list(paths); self.labels = list(labels); self.tf = tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.tf(img), int(self.labels[i])

def make_loaders(trp, trl, vap, val):
    tr_ds = FERDataset(trp, trl, train_tf)
    va_ds = FERDataset(vap, val, eval_tf)
    trl = np.asarray(trl)
    cls_count = np.bincount(trl, minlength=NUM_CLASSES).astype(np.float32)
    cls_w = 1.0 / np.maximum(cls_count, 1.0)
    samp_w = cls_w[trl]
    sampler = WeightedRandomSampler(samp_w, num_samples=len(trl), replacement=True)
    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=4,
                           pin_memory=True, persistent_workers=True, prefetch_factor=4)
    va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4,
                           pin_memory=True, persistent_workers=True, prefetch_factor=4)
    return tr_loader, va_loader, cls_w

## Model (timm backbone, pretrained)

In [ ]:
import timm, torch.nn as nn
device = "cuda" if torch.cuda.is_available() else "cpu"
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES).to(device)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print("%s | %.2f M params" % (MODEL_NAME, n_params))

## Train - Stratified 3-Fold CV (best model by **val_acc**)
3-fold CV on the 75% pool. The sampler balances each batch, so the loss is
plain label-smoothed CE (no double-balancing). Each fold checkpoints on the
**highest val_acc** and early-stops on patience; the best-val_acc fold is saved.
The next cell then refits on the full pool for the deployment checkpoint.

In [ ]:
import numpy as np, torch, torch.nn as nn, timm, copy
from sklearn.model_selection import StratifiedKFold
device = "cuda" if torch.cuda.is_available() else "cpu"
N_SPLITS = 3            # lighter CV (image CNN x ~35k imgs) - fits the Colab budget
PATIENCE = 10          # early stop on val_acc (no improvement for this many epochs)

# Stress = weighted sum of negative-affect emotion probabilities (saved in ckpt).
STRESS_WEIGHTS = {"fear": 1.0, "angry": 0.7, "sad": 0.6, "disgust": 0.4,
                  "surprise": 0.2, "happy": 0.0, "neutral": 0.0}
save_path = os.path.join(WORK_DIR, "emotion_fer_best.pt")

def run_eval(model, loader, criterion):
    model.eval(); vloss = 0.0; vc = vn = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x); loss = criterion(out, y)
            vloss += loss.item() * y.size(0)
            vc += (out.argmax(1) == y).sum().item(); vn += y.size(0)
    return vloss / max(vn, 1), vc / max(vn, 1)

def train_one_fold(trp, trl, vap, val, fold):
    tr_loader, va_loader, _ = make_loaders(trp, trl, vap, val)
    model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES).to(device)
    # The WeightedRandomSampler already balances each batch, so use plain CE.
    # (Sampler + class-weighted loss together = double-balancing -> hurts overall acc.)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda") if device == "cuda" else None
    best_va = -1.0; best_vloss = float("inf"); best_state = None; bad = 0
    for epoch in range(EPOCHS):
        model.train(); tc = tn = 0; run = 0.0
        for x, y in tr_loader:
            x, y = x.to(device), y.to(device); opt.zero_grad()
            if scaler:
                with torch.amp.autocast("cuda"):
                    out = model(x); loss = criterion(out, y)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else:
                out = model(x); loss = criterion(out, y); loss.backward(); opt.step()
            run += loss.item() * y.size(0)
            tc += (out.argmax(1) == y).sum().item(); tn += y.size(0)
        sched.step()
        vloss, va = run_eval(model, va_loader, criterion)
        print("  [fold %d] ep %02d/%d  loss=%.4f train_acc=%.3f  val_loss=%.4f val_acc=%.3f" % (
            fold, epoch + 1, EPOCHS, run / max(tn, 1), tc / max(tn, 1), vloss, va))
        # select on val_acc (tie-break: lower val_loss)
        if va > best_va + 1e-4 or (abs(va - best_va) <= 1e-4 and vloss < best_vloss):
            best_va = va; best_vloss = vloss; best_state = copy.deepcopy(model.state_dict()); bad = 0
        else:
            bad += 1
            if bad >= PATIENCE:
                print("  [fold %d] early stop @ epoch %d (best val_acc=%.4f)" % (fold, epoch + 1, best_va))
                break
    return best_state, best_vloss, best_va

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
fold_vloss, fold_vacc = [], []
global_best_va = -1.0; global_best_state = None
for fold, (a, b) in enumerate(skf.split(tr_paths, tr_lab), 1):
    print("Fold %d/%d  train=%d  val=%d" % (fold, N_SPLITS, len(a), len(b)))
    state, vloss, va = train_one_fold(tr_paths[a], tr_lab[a], tr_paths[b], tr_lab[b], fold)
    fold_vloss.append(vloss); fold_vacc.append(va)
    if va > global_best_va:
        global_best_va = va; global_best_state = state
        print("  ** new global best val_acc=%.4f (fold %d) **" % (va, fold))

print("\nCV val_acc : %.4f +/- %.4f" % (float(np.mean(fold_vacc)), float(np.std(fold_vacc))))
print("CV val_loss: %.4f +/- %.4f" % (float(np.mean(fold_vloss)), float(np.std(fold_vloss))))
torch.save({"model_state_dict": global_best_state, "class_names": CLASS_NAMES,
    "val_acc": float(global_best_va), "selection": "val_acc",
    "cv_val_acc_mean": float(np.mean(fold_vacc)), "cv_val_loss_mean": float(np.mean(fold_vloss)),
    "config": {"model_name": MODEL_NAME, "img_size": IMG_SIZE,
        "mean": MEAN, "std": STD, "grayscale_3ch": True,
        "stress_weights": STRESS_WEIGHTS}}, save_path)
print("Saved best-fold model (val_acc=%.4f) ->" % global_best_va, save_path)


## Final fit - refit on the FULL train+val pool (deployment model)
Cross-validation above is for *measuring* accuracy. The model you ship should use
all the data, so this trains one model on the entire 75% pool (10% held back only
for early-stopping on val_acc) and overwrites `emotion_fer_best.pt`. The 25%
held-out test in Evaluate stays untouched, so that number is still honest.

In [ ]:
# Final deployment model: refit on the FULL train+val pool (more data = +acc).
# CV above measured generalization; THIS is the checkpoint you actually ship.
# A small internal 90/10 stratified split is used only to early-stop on val_acc.
# The 25% held-out TEST set (scored in Evaluate) is never touched here.
import numpy as np, torch, torch.nn as nn, timm, copy
from sklearn.model_selection import train_test_split
device = "cuda" if torch.cuda.is_available() else "cpu"
FINAL_PATIENCE = 10

ftp, fvp, ftl, fvl = train_test_split(
    tr_paths, tr_lab, test_size=0.10, random_state=SEED, stratify=tr_lab)
tr_loader, va_loader, _ = make_loaders(ftp, ftl, fvp, fvl)
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler("cuda") if device == "cuda" else None
best_va = -1.0; best_state = None; bad = 0
for epoch in range(EPOCHS):
    model.train()
    for x, y in tr_loader:
        x, y = x.to(device), y.to(device); opt.zero_grad()
        if scaler:
            with torch.amp.autocast("cuda"):
                out = model(x); loss = criterion(out, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        else:
            out = model(x); loss = criterion(out, y); loss.backward(); opt.step()
    sched.step()
    vloss, va = run_eval(model, va_loader, criterion)
    print("  [final] ep %02d/%d  val_loss=%.4f val_acc=%.3f" % (epoch + 1, EPOCHS, vloss, va))
    if va > best_va + 1e-4:
        best_va = va; best_state = copy.deepcopy(model.state_dict()); bad = 0
    else:
        bad += 1
        if bad >= FINAL_PATIENCE:
            print("  [final] early stop @ epoch %d (best val_acc=%.4f)" % (epoch + 1, best_va)); break

torch.save({"model_state_dict": best_state, "class_names": CLASS_NAMES,
    "val_acc": float(best_va), "selection": "val_acc_fullpool",
    "cv_val_acc_mean": float(np.mean(fold_vacc)), "cv_val_loss_mean": float(np.mean(fold_vloss)),
    "config": {"model_name": MODEL_NAME, "img_size": IMG_SIZE,
        "mean": MEAN, "std": STD, "grayscale_3ch": True,
        "stress_weights": STRESS_WEIGHTS}}, save_path)
print("Saved FULL-POOL deployment model (val_acc=%.4f) -> %s" % (best_va, save_path))


## Evaluate - per-class precision / recall + a stress demo
Negative-affect emotions (fear/angry/sad) are what feed the stress score, so check
their recall. The stress score below is exactly what the app will compute.

In [ ]:
import numpy as np, torch, timm
from torch.utils.data import DataLoader
device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt = torch.load(os.path.join(WORK_DIR, "emotion_fer_best.pt"), map_location=device)
model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES).to(device)
model.load_state_dict(ckpt["model_state_dict"]); model.eval()
sw = ckpt["config"]["stress_weights"]
stress_vec = np.array([sw.get(c, 0.0) for c in CLASS_NAMES], dtype=np.float32)
test_loader = DataLoader(FERDataset(te_paths, te_lab, eval_tf),
                         batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int); stress_vals = []
with torch.no_grad():
    for x, y in test_loader:
        prob = torch.softmax(model(x.to(device)), 1).cpu().numpy()
        p = prob.argmax(1); y = y.numpy()
        stress_vals.extend((prob * stress_vec).sum(1).tolist())
        for t, pp in zip(y, p): cm[t, pp] += 1
print("HELD-OUT TEST (25%)  -  n =", int(cm.sum()))
print("%-10s %6s %6s %6s" % ("class", "prec", "rec", "n"))
for i, c in enumerate(CLASS_NAMES):
    tp = cm[i, i]; fp = cm[:, i].sum() - tp; fn = cm[i, :].sum() - tp
    print("%-10s %6.3f %6.3f %6d" % (c, tp / max(tp + fp, 1), tp / max(tp + fn, 1), cm[i, :].sum()))
print("\nTest acc:", round(float(np.trace(cm)) / max(int(cm.sum()), 1), 4),
      "| CV val_loss mean:", round(ckpt["cv_val_loss_mean"], 4))
print("Stress score (test) mean=%.3f  p90=%.3f  (alert when > ~0.5)" % (
    float(np.mean(stress_vals)), float(np.percentile(stress_vals, 90))))

## Export & wire back
`emotion_fer_best.pt` is in `WORK_DIR` on Drive.
1. Download it into the repo at `models/weights/emotion_fer_best.pt`.
2. The checkpoint carries `class_names`, `img_size`, normalization, and the
   `stress_weights` mapping - everything inference needs.
3. Ping me and I'll swap `stages/emotion_analysis.py` off DeepFace: crop the face
   (existing face pipeline) -> this model -> softmax over 7 emotions -> the same
   `stress_weights` dot-product for a continuous stress score (replacing the
   brittle eye/brow geometry heuristic).

**Accuracy upgrade:** add RAF-DB / AffectNet face crops to the train folders for
a few extra points, or bump `IMG_SIZE`/`MODEL_NAME` in CONFIG.